In [5]:
#@title Install DEAP
!pip install deap --quiet

In [6]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

# Define the fitness class (single objective maximization)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,)) # if 1.0, maxmization
creator.create("Individual", list, fitness=creator.FitnessMin)

# Define a sphere function to be minimized
def sphere(x):
    return x[0]**2 + x[1]**2,

def polynomial(x):  # minima at y=0
    return (x[0] ** 3 - 2 * x[0] ** 2 - x[0] + 2) ** 2,

def rosenbrock(x):
    return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2,

def booth(x): # minima at (1,3)
    return ((x[0] + 2 * x[1] - 7) ** 2 + (2 * x[0] + x[1] - 5) ** 2, )

def ackley(x): # minima at (0,0)
    a = 20
    b = 0.2
    c = 2 * np.pi
    d = 2
    sum1 = -a * np.exp(-b * np.sqrt((1 / d) * (x[0] ** 2 + x[1] ** 2)))
    sum2 = -np.exp((1 / d) * (np.cos(c * x[0]) + np.cos(c * x[1])))
    return (sum1 + sum2 + a + np.exp(1),) # return as a tuple

/usr/local/lib/python3.10/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.10/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [7]:
def deap_run(eval_func, numVar):
  # Create the toolbox
  toolbox = base.Toolbox()
  toolbox.register("attr_float", random.uniform, -5, 5)
  toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=numVar)
  toolbox.register("population", tools.initRepeat, list, toolbox.individual)
  toolbox.register("evaluate", eval_func)
  if numVar > 1:
    toolbox.register("mate", tools.cxTwoPoint)  # Use cxTwoPoint for numVar > 1
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
  else:
    toolbox.register("mate", tools.cxUniform, indpb=0.5) # added this for single gene
    #toolbox.register("mate", tools.cxOnePoint) # this also works
    # Use a suitable operator for numVar = 1, e.g., mutGaussian
    toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.1)
  toolbox.register("select", tools.selTournament, tournsize=3)

  # Run the algorithm
  successes = 0
  trials = 3
  for tr in range(trials):
    population = toolbox.population(n=300)
    # Run the algorithm
    final_population = algorithms.eaSimple(
        population,
        toolbox,
        cxpb=0.5,  # Crossover probability
        mutpb=0.2,  # Mutation probability
        ngen=50,    # Number of generations
        verbose=False,
    )

    # Get the best individual
    best_ind = tools.selBest(final_population[0], 1)[0]
    print(f"---------- Trial: {tr} ")
    print("Best individual:", best_ind)
    if best_ind.fitness.values[0] < 0.00005:
      successes += 1
    print("Best fitness:", best_ind.fitness.values[0])
  print(f"\nSystem Success %: {successes*100/trials}")

########################### M A I N ###########################
print("\n***************** Sphere Function *****************")
deap_run(sphere, 2)

print("\n***************** Polynomial Function *****************")
deap_run(polynomial, 1)

print("\n***************** Rosenbrock Function *****************")
deap_run(rosenbrock, 2)

print("\n***************** Booth Function *****************")
deap_run(booth, 2)

print("\n***************** Ackley Function *****************")
deap_run(ackley, 2)



***************** Sphere Function *****************
---------- Trial: 0 
Best individual: [0.0, 0.0]
Best fitness: 0.0
---------- Trial: 1 
Best individual: [0.0, 0.0]
Best fitness: 0.0
---------- Trial: 2 
Best individual: [0.0, 0.0]
Best fitness: 0.0

System Success %: 100.0

***************** Polynomial Function *****************
---------- Trial: 0 
Best individual: [1.0020631807098592]
Best fitness: 1.6991674795981774e-05
---------- Trial: 1 
Best individual: [0.9992323635168296]
Best fitness: 2.358871406555026e-06
---------- Trial: 2 
Best individual: [1.9992975566743523]
Best fitness: 4.432526484933419e-06

System Success %: 100.0

***************** Rosenbrock Function *****************
---------- Trial: 0 
Best individual: [0.9925587021931541, 1.0]
Best fitness: 0.022040026214492414
---------- Trial: 1 
Best individual: [0.904890738536837, 0.8048567814826688]
Best fitness: 0.02856316701445994
---------- Trial: 2 
Best individual: [0.7572397511755629, 0.5587715279694443]
Best f